# 第12章 信用风险与定价模型 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch12_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch12_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：例12.1 + 图12-1（PD/利差随杠杆）


In [ ]:
import numpy as np
from fi import credit as cr, plotting
plotting.use_chinese_style()
for V in (120,150,200):
    r = cr.merton_pd(V,100,0.25,0.03,1)
    print(f'V={V}: DD={r["distance_to_default"]:.2f} PD={r["pd"]*100:.2f}% 利差={cr.merton_credit_spread(V,100,0.25,0.03,1)*1e4:.0f}bp')
lev = np.linspace(0.4,0.92,40); D=100.0; Vs=D/lev
fig, ax1 = plotting.new_axes()
ax1.plot(lev, [cr.merton_pd(v,D,0.25,0.03,1)['pd']*100 for v in Vs], 'C0'); ax1.set_xlabel('杠杆 D/V'); ax1.set_ylabel('PD (%)', color='C0')
ax2 = ax1.twinx(); ax2.plot(lev, [cr.merton_credit_spread(v,D,0.25,0.03,1)*1e4 for v in Vs], 'C3--'); ax2.set_ylabel('利差 (bp)', color='C3')
ax1.set_title('Merton: PD与利差随杠杆上升'); fig.tight_layout()


## 编程实验 8：隐含违约率曲线 + 回收率敏感性


In [ ]:
tenors = [1,2,3,5,7,10]; spreads = [0.005,0.008,0.011,0.015,0.018,0.022]
fig, ax = plotting.new_axes()
for R in (0.30,0.40,0.50):
    t, pd = cr.implied_default_curve(tenors, spreads, recovery=R)
    ax.plot(t, pd*100, marker='o', label=f'回收率{R:.0%}')
    if R==0.40:
        for ti,pdi in zip(t,pd): print(f'{ti:.0f}yr 累计违约(R=40%)={pdi*100:.2f}%')
ax.set_xlabel('期限(年)'); ax.set_ylabel('累计违约概率 (%)'); ax.set_title('隐含违约率曲线'); ax.legend(); fig.tight_layout()


## 编程实验 9：QuantLib FlatHazardRate 验证


In [ ]:
import QuantLib as ql
today = ql.Date(15,6,2026); ql.Settings.instance().evaluationDate = today
lam = cr.hazard_from_spread(0.02, 0.4)
dpts = ql.FlatHazardRate(today, ql.QuoteHandle(ql.SimpleQuote(lam)), ql.Actual365Fixed())
for y in (1,3,5):
    d = today+ql.Period(y,ql.Years)
    print(f'{y}yr: fi生存={cr.survival_probability(lam,y)*100:.2f}% QL生存={dpts.survivalProbability(d)*100:.2f}%')
